# Module 5 — Automated Business Insight Generation

This notebook converts cleaned and engineered sales data into
structured business insights that can later be passed to an LLM.

## Objectives

- Generate KPI summaries
- Detect sales and profit trends
- Identify profitability problems
- Detect high-return categories
- Identify top and bottom performers
- Generate structured insights
- Prepare outputs for LLM integration

In [1]:
import os

os.getcwd()

'C:\\Users\\LENOVO\\AI_Data_Insights_Generator'

In [2]:
os.listdir()

['     02_Data_Cleaning.ipynb',
 '.gitignore',
 '.ipynb_checkpoints',
 '01_Data_Exploration.ipynb',
 '03_Exploratory_Data_Analysis.ipynb',
 '04_Insight_Generation.ipynb',
 '05_Final_Report.ipynb',
 'charts',
 'data',
 'notebooks',
 'README.md',
 'reports',
 'SampleSuperstore.csv',
 'screenshots',
 'src',
 'venv']

In [3]:
import os

os.listdir("data")

['SampleSuperstore.csv', 'SampleSuperstore_Cleaned.csv']

In [4]:
import pandas as pd

df = pd.read_csv("data/SampleSuperstore_Cleaned.csv")

df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,product_id,category,sub_category,product_name,sales,quantity,profit,payment_mode,returned,shipping_days
0,4918,CA-2019-160304,2019-01-01,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,FUR-BO-10004709,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",73.94,1,28.2668,Online,0.0,6
1,4919,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,FUR-BO-10004709,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Medium Ch...",173.94,3,38.2668,Online,0.0,5
2,4920,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,TEC-PH-10000455,Technology,Phones,GE 30522EE2,231.98,2,67.2742,Cards,0.0,5
3,3074,CA-2019-125206,2019-01-03,2019-01-05,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,...,OFF-ST-10003692,Office Supplies,Storage,Recycled Steel Personal File for Hanging File ...,114.46,2,28.6150,Online,0.0,2
4,8604,US-2019-116365,2019-01-03,2019-01-08,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,TEC-AC-10002217,Technology,Accessories,Imation Clip USB flash drive - 8 GB,30.08,2,-5.2640,Online,0.0,5


In [5]:
df.shape

(5901, 22)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5901 entries, 0 to 5900
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         5901 non-null   int64  
 1   order_id       5901 non-null   str    
 2   order_date     5901 non-null   str    
 3   ship_date      5901 non-null   str    
 4   ship_mode      5901 non-null   str    
 5   customer_id    5901 non-null   str    
 6   customer_name  5901 non-null   str    
 7   segment        5901 non-null   str    
 8   country        5901 non-null   str    
 9   city           5901 non-null   str    
 10  state          5901 non-null   str    
 11  region         5901 non-null   str    
 12  product_id     5901 non-null   str    
 13  category       5901 non-null   str    
 14  sub_category   5901 non-null   str    
 15  product_name   5901 non-null   str    
 16  sales          5901 non-null   float64
 17  quantity       5901 non-null   int64  
 18  profit         5901

In [7]:
df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'profit',
 'payment_mode',
 'returned',
 'shipping_days']

In [8]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv("data/SampleSuperstore_Cleaned.csv")

# Convert dates
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

# Time features
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
df["order_year"] = df["order_date"].dt.year
df["order_quarter"] = df["order_date"].dt.quarter

# Business metrics
df["profit_margin"] = (
    df["profit"] / df["sales"] * 100
)

df["sales_per_quantity"] = (
    df["sales"] / df["quantity"]
)

df["profit_per_quantity"] = (
    df["profit"] / df["quantity"]
)

df["is_profitable"] = df["profit"] > 0

df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,payment_mode,returned,shipping_days,order_month,order_year,order_quarter,profit_margin,sales_per_quantity,profit_per_quantity,is_profitable
0,4918,CA-2019-160304,2019-01-01,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Online,0.0,6,2019-01,2019,1,38.229375,73.94,28.2668,True
1,4919,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Online,0.0,5,2019-01,2019,1,22.000000,57.98,12.7556,True
2,4920,CA-2019-160304,2019-01-02,2019-01-07,Standard Class,BM-11575,Brendan Murry,Corporate,United States,Gaithersburg,...,Cards,0.0,5,2019-01,2019,1,29.000000,115.99,33.6371,True
3,3074,CA-2019-125206,2019-01-03,2019-01-05,First Class,LR-16915,Lena Radford,Consumer,United States,Los Angeles,...,Online,0.0,2,2019-01,2019,1,25.000000,57.23,14.3075,True
4,8604,US-2019-116365,2019-01-03,2019-01-08,Standard Class,CA-12310,Christine Abelman,Corporate,United States,San Antonio,...,Online,0.0,5,2019-01,2019,1,-17.500000,15.04,-2.6320,False


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5901 entries, 0 to 5900
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   row_id               5901 non-null   int64         
 1   order_id             5901 non-null   str           
 2   order_date           5901 non-null   datetime64[us]
 3   ship_date            5901 non-null   datetime64[us]
 4   ship_mode            5901 non-null   str           
 5   customer_id          5901 non-null   str           
 6   customer_name        5901 non-null   str           
 7   segment              5901 non-null   str           
 8   country              5901 non-null   str           
 9   city                 5901 non-null   str           
 10  state                5901 non-null   str           
 11  region               5901 non-null   str           
 12  product_id           5901 non-null   str           
 13  category             5901 non-null   str    

First automated insight: KPI summary

In [10]:
def generate_kpi_summary(df):
    return {
        "Total Sales": round(df["sales"].sum(), 2),
        "Total Profit": round(df["profit"].sum(), 2),
        "Total Quantity": int(df["quantity"].sum()),
        "Total Orders": int(df["order_id"].nunique()),
        "Average Profit Margin (%)": round(
            df["profit"].sum() / df["sales"].sum() * 100, 2
        ),
        "Return Rate (%)": round(
            df["returned"].sum() / df["order_id"].nunique() * 100, 2
        )
    }

In [11]:
kpi_summary = generate_kpi_summary(df)

kpi_summary

{'Total Sales': np.float64(1565804.32),
 'Total Profit': np.float64(175262.11),
 'Total Quantity': 22317,
 'Total Orders': 3003,
 'Average Profit Margin (%)': np.float64(11.19),
 'Return Rate (%)': np.float64(9.56)}

In [12]:
df["returned"].value_counts()

returned
0.0    5614
1.0     287
Name: count, dtype: int64

In [13]:
def generate_kpi_summary(df):
    return {
        "Total Sales": round(df["sales"].sum(), 2),
        "Total Profit": round(df["profit"].sum(), 2),
        "Total Quantity": int(df["quantity"].sum()),
        "Total Orders": int(df["order_id"].nunique()),
        "Average Profit Margin (%)": round(
            df["profit"].sum() / df["sales"].sum() * 100, 2
        ),
        "Return Rate (%)": round(
            df["returned"].mean() * 100, 2
        )
    }

In [14]:
kpi_summary = generate_kpi_summary(df)

kpi_summary

{'Total Sales': np.float64(1565804.32),
 'Total Profit': np.float64(175262.11),
 'Total Quantity': 22317,
 'Total Orders': 3003,
 'Average Profit Margin (%)': np.float64(11.19),
 'Return Rate (%)': np.float64(4.86)}

# Profitability Insight

In [15]:
def generate_profitability_insight(kpi):
    margin = kpi["Average Profit Margin (%)"]

    if margin < 5:
        status = "critical"
        message = (
            f"Overall profit margin is {margin:.2f}%, "
            "indicating a critical profitability concern."
        )
    elif margin < 10:
        status = "warning"
        message = (
            f"Overall profit margin is {margin:.2f}%, "
            "indicating profitability should be monitored."
        )
    else:
        status = "healthy"
        message = (
            f"Overall profit margin is {margin:.2f}%, "
            "indicating a relatively healthy overall profitability level."
        )

    return {
        "insight_type": "profitability",
        "status": status,
        "message": message
    }

In [16]:
profitability_insight = generate_profitability_insight(kpi_summary)

profitability_insight

{'insight_type': 'profitability',
 'status': 'healthy',
 'message': 'Overall profit margin is 11.19%, indicating a relatively healthy overall profitability level.'}

# Year-over-Year Insight

In [17]:
def generate_yearly_insights(df):

    yearly = (
        df.groupby("order_year")
        .agg(
            Sales=("sales", "sum"),
            Profit=("profit", "sum"),
            Quantity=("quantity", "sum"),
            Orders=("order_id", "nunique")
        )
    )

    yearly["Profit_Margin_%"] = (
        yearly["Profit"] / yearly["Sales"] * 100
    )

    years = sorted(yearly.index)

    if len(years) < 2:
        return []

    previous = years[-2]
    current = years[-1]

    insights = []

    for metric in ["Sales", "Profit", "Quantity", "Orders"]:
        old_value = yearly.loc[previous, metric]
        new_value = yearly.loc[current, metric]

        change_pct = (
            (new_value - old_value) / old_value * 100
        )

        insights.append({
            "insight_type": "yearly_change",
            "metric": metric,
            "from_year": int(previous),
            "to_year": int(current),
            "change_percent": round(change_pct, 2)
        })

    # Profit margin change
    old_margin = yearly.loc[previous, "Profit_Margin_%"]
    new_margin = yearly.loc[current, "Profit_Margin_%"]

    margin_change = new_margin - old_margin

    insights.append({
        "insight_type": "margin_change",
        "from_year": int(previous),
        "to_year": int(current),
        "margin_change_pp": round(margin_change, 2)
    })

    return insights

In [18]:
yearly_insights = generate_yearly_insights(df)

yearly_insights

[{'insight_type': 'yearly_change',
  'metric': 'Sales',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(77.29)},
 {'insight_type': 'yearly_change',
  'metric': 'Profit',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(14.2)},
 {'insight_type': 'yearly_change',
  'metric': 'Quantity',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(26.84)},
 {'insight_type': 'yearly_change',
  'metric': 'Orders',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(28.37)},
 {'insight_type': 'margin_change',
  'from_year': 2019,
  'to_year': 2020,
  'margin_change_pp': np.float64(-5.16)}]

# Automatically classify the yearly changes

In [19]:
def classify_yearly_insights(insights):

    classified = []

    for insight in insights:

        item = insight.copy()

        if insight["insight_type"] == "yearly_change":

            change = insight["change_percent"]

            if insight["metric"] == "Profit":
                if change < 0:
                    item["severity"] = "critical"
                    item["interpretation"] = "Profit declined."
                elif change < 10:
                    item["severity"] = "warning"
                    item["interpretation"] = "Profit growth is weak."
                else:
                    item["severity"] = "positive"
                    item["interpretation"] = "Profit increased."

            elif insight["metric"] == "Sales":

                if change < 0:
                    item["severity"] = "negative"
                    item["interpretation"] = "Sales declined."
                else:
                    item["severity"] = "positive"
                    item["interpretation"] = "Sales increased."

            else:

                if change < 0:
                    item["severity"] = "negative"
                    item["interpretation"] = (
                        f"{insight['metric']} declined."
                    )
                else:
                    item["severity"] = "positive"
                    item["interpretation"] = (
                        f"{insight['metric']} increased."
                    )

        elif insight["insight_type"] == "margin_change":

            change = insight["margin_change_pp"]

            if change <= -5:
                item["severity"] = "critical"
                item["interpretation"] = (
                    "Profit margin deteriorated significantly."
                )

            elif change < 0:
                item["severity"] = "warning"
                item["interpretation"] = (
                    "Profit margin declined."
                )

            else:
                item["severity"] = "positive"
                item["interpretation"] = (
                    "Profit margin improved."
                )

        classified.append(item)

    return classified

In [20]:
classified_yearly_insights = classify_yearly_insights(
    yearly_insights
)

classified_yearly_insights

[{'insight_type': 'yearly_change',
  'metric': 'Sales',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(77.29),
  'severity': 'positive',
  'interpretation': 'Sales increased.'},
 {'insight_type': 'yearly_change',
  'metric': 'Profit',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(14.2),
  'severity': 'positive',
  'interpretation': 'Profit increased.'},
 {'insight_type': 'yearly_change',
  'metric': 'Quantity',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(26.84),
  'severity': 'positive',
  'interpretation': 'Quantity increased.'},
 {'insight_type': 'yearly_change',
  'metric': 'Orders',
  'from_year': 2019,
  'to_year': 2020,
  'change_percent': np.float64(28.37),
  'severity': 'positive',
  'interpretation': 'Orders increased.'},
 {'insight_type': 'margin_change',
  'from_year': 2019,
  'to_year': 2020,
  'margin_change_pp': np.float64(-5.16),
  'severity': 'critical',
  'interpretation': 'Profit margin deteri

Generate human-readable insight messages

In [21]:
def create_insight_messages(classified_insights):

    messages = []

    for insight in classified_insights:

        if insight["insight_type"] == "yearly_change":

            metric = insight["metric"]
            change = insight["change_percent"]
            from_year = insight["from_year"]
            to_year = insight["to_year"]

            if change >= 0:
                direction = "increased"
            else:
                direction = "decreased"

            message = (
                f"{metric} {direction} by "
                f"{abs(change):.2f}% from {from_year} to {to_year}."
            )

        elif insight["insight_type"] == "margin_change":

            change = insight["margin_change_pp"]
            from_year = insight["from_year"]
            to_year = insight["to_year"]

            if change < 0:
                message = (
                    f"Profit margin decreased by "
                    f"{abs(change):.2f} percentage points "
                    f"from {from_year} to {to_year}."
                )
            else:
                message = (
                    f"Profit margin increased by "
                    f"{change:.2f} percentage points "
                    f"from {from_year} to {to_year}."
                )

        messages.append({
            "severity": insight["severity"],
            "message": message
        })

    return messages

In [22]:
insight_messages = create_insight_messages(
    classified_yearly_insights
)

insight_messages

[{'severity': 'positive',
  'message': 'Sales increased by 77.29% from 2019 to 2020.'},
 {'severity': 'positive',
  'message': 'Profit increased by 14.20% from 2019 to 2020.'},
 {'severity': 'positive',
  'message': 'Quantity increased by 26.84% from 2019 to 2020.'},
 {'severity': 'positive',
  'message': 'Orders increased by 28.37% from 2019 to 2020.'},
 {'severity': 'critical',
  'message': 'Profit margin decreased by 5.16 percentage points from 2019 to 2020.'}]

# Category Performance Insights

In [23]:
def generate_category_insights(df):

    category_yearly = (
        df.groupby(["order_year", "category"])
        .agg(
            Sales=("sales", "sum"),
            Profit=("profit", "sum")
        )
        .reset_index()
    )

    category_yearly["Profit_Margin_%"] = (
        category_yearly["Profit"]
        / category_yearly["Sales"]
        * 100
    )

    insights = []

    for category in category_yearly["category"].unique():

        category_data = category_yearly[
            category_yearly["category"] == category
        ].sort_values("order_year")

        if len(category_data) < 2:
            continue

        previous = category_data.iloc[-2]
        current = category_data.iloc[-1]

        sales_change = (
            (current["Sales"] - previous["Sales"])
            / previous["Sales"] * 100
        )

        profit_change = (
            (current["Profit"] - previous["Profit"])
            / previous["Profit"] * 100
        )

        margin_change = (
            current["Profit_Margin_%"]
            - previous["Profit_Margin_%"]
        )

        insights.append({
            "category": category,
            "from_year": int(previous["order_year"]),
            "to_year": int(current["order_year"]),
            "sales_change_percent": round(sales_change, 2),
            "profit_change_percent": round(profit_change, 2),
            "margin_change_pp": round(margin_change, 2)
        })

    return insights

In [24]:
category_insights = generate_category_insights(df)

category_insights

[{'category': 'Furniture',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(31.22),
  'profit_change_percent': np.float64(-56.81),
  'margin_change_pp': np.float64(-2.4)},
 {'category': 'Office Supplies',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(167.96),
  'profit_change_percent': np.float64(13.33),
  'margin_change_pp': np.float64(-11.57)},
 {'category': 'Technology',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(41.99),
  'profit_change_percent': np.float64(27.43),
  'margin_change_pp': np.float64(-2.1)}]

# Classify category risks

In [25]:
def classify_category_insights(category_insights):

    classified = []

    for insight in category_insights:

        item = insight.copy()

        margin_change = insight["margin_change_pp"]
        profit_change = insight["profit_change_percent"]

        # Critical: profit declined substantially
        if profit_change < -20:
            item["severity"] = "critical"
            item["interpretation"] = (
                "Profit declined significantly despite category sales activity."
            )

        # Critical: major margin deterioration
        elif margin_change <= -10:
            item["severity"] = "critical"
            item["interpretation"] = (
                "Profit margin deteriorated significantly."
            )

        # Warning: moderate margin deterioration
        elif margin_change < -2:
            item["severity"] = "warning"
            item["interpretation"] = (
                "Profit margin declined and should be monitored."
            )

        else:
            item["severity"] = "positive"
            item["interpretation"] = (
                "Category profitability remained relatively stable."
            )

        classified.append(item)

    return classified

In [26]:
classified_category_insights = classify_category_insights(
    category_insights
)

classified_category_insights

[{'category': 'Furniture',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(31.22),
  'profit_change_percent': np.float64(-56.81),
  'margin_change_pp': np.float64(-2.4),
  'severity': 'critical',
  'interpretation': 'Profit declined significantly despite category sales activity.'},
 {'category': 'Office Supplies',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(167.96),
  'profit_change_percent': np.float64(13.33),
  'margin_change_pp': np.float64(-11.57),
  'severity': 'critical',
  'interpretation': 'Profit margin deteriorated significantly.'},
 {'category': 'Technology',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(41.99),
  'profit_change_percent': np.float64(27.43),
  'margin_change_pp': np.float64(-2.1),
  'severity': 'warning',
  'interpretation': 'Profit margin declined and should be monitored.'}]

Build the Top Business Risks engine

In [27]:
def generate_top_business_risks(
    yearly_insights,
    classified_category_insights,
    top_n=10
):
    
    risks = []

    # -----------------------------
    # Year-level risks
    # -----------------------------
    for insight in yearly_insights:

        if insight["insight_type"] == "margin_change":

            change = insight["margin_change_pp"]

            if change < 0:
                risks.append({
                    "level": "year",
                    "area": "Overall Business",
                    "risk_type": "Margin Decline",
                    "severity": (
                        "critical" if change <= -5 else "warning"
                    ),
                    "score": abs(change),
                    "message": (
                        f"Overall profit margin declined by "
                        f"{abs(change):.2f} percentage points."
                    )
                })

        elif insight["metric"] == "Profit":

            change = insight["change_percent"]

            if change < 0:
                risks.append({
                    "level": "year",
                    "area": "Overall Business",
                    "risk_type": "Profit Decline",
                    "severity": "critical",
                    "score": abs(change),
                    "message": (
                        f"Overall profit declined by "
                        f"{abs(change):.2f}%."
                    )
                })

    # -----------------------------
    # Category-level risks
    # -----------------------------
    for insight in classified_category_insights:

        if insight["severity"] in ["critical", "warning"]:

            risks.append({
                "level": "category",
                "area": insight["category"],
                "risk_type": "Category Profitability",
                "severity": insight["severity"],
                "score": (
                    abs(insight["margin_change_pp"])
                    + max(0, -insight["profit_change_percent"] / 10)
                ),
                "message": (
                    f"{insight['category']} experienced a "
                    f"{insight['margin_change_pp']:.2f} percentage-point "
                    f"margin change and "
                    f"{insight['profit_change_percent']:.2f}% "
                    f"profit change."
                )
            })

    # -----------------------------
    # Rank risks
    # -----------------------------
    risks = sorted(
        risks,
        key=lambda x: x["score"],
        reverse=True
    )

    return risks[:top_n]

In [28]:
top_business_risks = generate_top_business_risks(
    yearly_insights,
    classified_category_insights
)

top_business_risks

[{'level': 'category',
  'area': 'Office Supplies',
  'risk_type': 'Category Profitability',
  'severity': 'critical',
  'score': np.float64(11.57),
  'message': 'Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.'},
 {'level': 'category',
  'area': 'Furniture',
  'risk_type': 'Category Profitability',
  'severity': 'critical',
  'score': np.float64(8.081),
  'message': 'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.'},
 {'level': 'year',
  'area': 'Overall Business',
  'risk_type': 'Margin Decline',
  'severity': 'critical',
  'score': np.float64(5.16),
  'message': 'Overall profit margin declined by 5.16 percentage points.'},
 {'level': 'category',
  'area': 'Technology',
  'risk_type': 'Category Profitability',
  'severity': 'warning',
  'score': np.float64(2.1),
  'message': 'Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.'}]

Generate the executive summary

In [29]:
def generate_executive_summary(kpi_summary, top_business_risks):

    critical_risks = [
        risk for risk in top_business_risks
        if risk["severity"] == "critical"
    ]

    warning_risks = [
        risk for risk in top_business_risks
        if risk["severity"] == "warning"
    ]

    summary = {
        "total_sales": kpi_summary["Total Sales"],
        "total_profit": kpi_summary["Total Profit"],
        "profit_margin": kpi_summary["Average Profit Margin (%)"],
        "return_rate": kpi_summary["Return Rate (%)"],
        "critical_risk_count": len(critical_risks),
        "warning_risk_count": len(warning_risks),
        "critical_risks": [
            risk["message"]
            for risk in critical_risks
        ],
        "warning_risks": [
            risk["message"]
            for risk in warning_risks
        ]
    }

    return summary

In [30]:
executive_summary = generate_executive_summary(
    kpi_summary,
    top_business_risks
)

executive_summary

{'total_sales': np.float64(1565804.32),
 'total_profit': np.float64(175262.11),
 'profit_margin': np.float64(11.19),
 'return_rate': np.float64(4.86),
 'critical_risk_count': 3,
 'warning_risk_count': 1,
 'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
  'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
  'Overall profit margin declined by 5.16 percentage points.'],
 'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']}

# sub-category risk detection

In [31]:
def generate_subcategory_insights(df):

    yearly = (
        df.groupby(["order_year", "sub_category"])
        .agg(
            Sales=("sales", "sum"),
            Profit=("profit", "sum")
        )
        .reset_index()
    )

    yearly["Profit_Margin_%"] = (
        yearly["Profit"] / yearly["Sales"] * 100
    )

    insights = []

    for sub_category in yearly["sub_category"].unique():

        data = yearly[
            yearly["sub_category"] == sub_category
        ].sort_values("order_year")

        if len(data) < 2:
            continue

        previous = data.iloc[-2]
        current = data.iloc[-1]

        sales_change = (
            (current["Sales"] - previous["Sales"])
            / previous["Sales"] * 100
        )

        profit_change = (
            (current["Profit"] - previous["Profit"])
            / abs(previous["Profit"]) * 100
            if previous["Profit"] != 0
            else np.nan
        )

        margin_change = (
            current["Profit_Margin_%"]
            - previous["Profit_Margin_%"]
        )

        insights.append({
            "sub_category": sub_category,
            "from_year": int(previous["order_year"]),
            "to_year": int(current["order_year"]),
            "sales_change_percent": round(sales_change, 2),
            "profit_change_percent": round(profit_change, 2),
            "margin_change_pp": round(margin_change, 2)
        })

    return insights

In [32]:
subcategory_insights = generate_subcategory_insights(df)

subcategory_insights

[{'sub_category': 'Accessories',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(91.92),
  'profit_change_percent': np.float64(62.17),
  'margin_change_pp': np.float64(-3.58)},
 {'sub_category': 'Appliances',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(120.58),
  'profit_change_percent': np.float64(48.36),
  'margin_change_pp': np.float64(-6.93)},
 {'sub_category': 'Art',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(651.6),
  'profit_change_percent': np.float64(57.14),
  'margin_change_pp': np.float64(-18.76)},
 {'sub_category': 'Binders',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(209.95),
  'profit_change_percent': np.float64(-24.92),
  'margin_change_pp': np.float64(-18.14)},
 {'sub_category': 'Bookcases',
  'from_year': 2019,
  'to_year': 2020,
  'sales_change_percent': np.float64(24.48),
  'profit_change_percent': np.float64(-342.43),
  'margin_change_pp'

# Sub-category Risk Engine

In [33]:
def generate_subcategory_risks(df, top_n=10):

    yearly = (
        df.groupby(["order_year", "sub_category"])
        .agg(
            Sales=("sales", "sum"),
            Profit=("profit", "sum"),
            Orders=("order_id", "nunique")
        )
        .reset_index()
    )

    yearly["Profit_Margin_%"] = (
        yearly["Profit"] / yearly["Sales"] * 100
    )

    risks = []

    for sub_category in yearly["sub_category"].unique():

        data = yearly[
            yearly["sub_category"] == sub_category
        ].sort_values("order_year")

        if len(data) < 2:
            continue

        previous = data.iloc[-2]
        current = data.iloc[-1]

        sales_change = (
            (current["Sales"] - previous["Sales"])
            / previous["Sales"] * 100
        )

        profit_change = (
            (current["Profit"] - previous["Profit"])
            / abs(previous["Profit"]) * 100
            if previous["Profit"] != 0
            else np.nan
        )

        margin_change = (
            current["Profit_Margin_%"]
            - previous["Profit_Margin_%"]
        )

        # Actual profit deterioration in dollars
        profit_change_dollars = (
            current["Profit"] - previous["Profit"]
        )

        # Risk score
        risk_score = 0

        # Profit loss has the strongest weight
        if profit_change_dollars < 0:
            risk_score += abs(profit_change_dollars)

        # Margin deterioration
        if margin_change < 0:
            risk_score += (
                abs(margin_change)
                * current["Sales"]
                / 100
            )

        # Classify severity
        if (
            profit_change_dollars < 0
            and margin_change <= -10
        ):
            severity = "critical"

        elif (
            profit_change_dollars < 0
            or margin_change <= -5
        ):
            severity = "warning"

        else:
            severity = "positive"

        if severity != "positive":

            risks.append({
                "sub_category": sub_category,
                "severity": severity,
                "risk_score": round(risk_score, 2),
                "sales_2020": round(current["Sales"], 2),
                "profit_2020": round(current["Profit"], 2),
                "sales_change_percent": round(
                    sales_change, 2
                ),
                "profit_change_percent": round(
                    profit_change, 2
                ),
                "margin_change_pp": round(
                    margin_change, 2
                ),
                "profit_change_dollars": round(
                    profit_change_dollars, 2
                )
            })

    risks = sorted(
        risks,
        key=lambda x: x["risk_score"],
        reverse=True
    )

    return risks[:top_n]

In [34]:
subcategory_risks = generate_subcategory_risks(df)

subcategory_risks

[{'sub_category': 'Binders',
  'severity': 'critical',
  'risk_score': np.float64(26539.05),
  'sales_2020': np.float64(132295.06),
  'profit_2020': np.float64(7669.74),
  'sales_change_percent': np.float64(209.95),
  'profit_change_percent': np.float64(-24.92),
  'margin_change_pp': np.float64(-18.14),
  'profit_change_dollars': np.float64(-2545.89)},
 {'sub_category': 'Paper',
  'severity': 'warning',
  'risk_score': np.float64(20536.65),
  'sales_2020': np.float64(77791.72),
  'profit_2020': np.float64(12040.84),
  'sales_change_percent': np.float64(259.12),
  'profit_change_percent': np.float64(32.73),
  'margin_change_pp': np.float64(-26.4),
  'profit_change_dollars': np.float64(2969.31)},
 {'sub_category': 'Machines',
  'severity': 'critical',
  'risk_score': np.float64(10890.68),
  'sales_2020': np.float64(40080.68),
  'profit_2020': np.float64(-2869.22),
  'sales_change_percent': np.float64(-22.78),
  'profit_change_percent': np.float64(-198.69),
  'margin_change_pp': np.float6

In [35]:
# Connect risks to products

In [36]:
def generate_product_risks(df, sub_category, top_n=10):

    data = df[
        df["sub_category"] == sub_category
    ].copy()

    yearly = (
        data.groupby(["order_year", "product_name"])
        .agg(
            Sales=("sales", "sum"),
            Profit=("profit", "sum"),
            Quantity=("quantity", "sum"),
            Orders=("order_id", "nunique")
        )
        .reset_index()
    )

    risks = []

    products = yearly["product_name"].unique()

    for product in products:

        product_data = yearly[
            yearly["product_name"] == product
        ].sort_values("order_year")

        if len(product_data) < 2:
            continue

        previous = product_data.iloc[-2]
        current = product_data.iloc[-1]

        profit_change = (
            current["Profit"] - previous["Profit"]
        )

        sales_change = (
            current["Sales"] - previous["Sales"]
        )

        if profit_change < 0:

            risks.append({
                "product_name": product,
                "profit_change": round(
                    profit_change, 2
                ),
                "sales_change": round(
                    sales_change, 2
                ),
                "profit_2020": round(
                    current["Profit"], 2
                ),
                "sales_2020": round(
                    current["Sales"], 2
                ),
                "quantity_2020": int(
                    current["Quantity"]
                ),
                "orders_2020": int(
                    current["Orders"]
                )
            })

    risks = sorted(
        risks,
        key=lambda x: x["profit_change"]
    )

    return risks[:top_n]

In [37]:
binder_product_risks = generate_product_risks(
    df,
    "Binders"
)

binder_product_risks

[{'product_name': 'Ibico EPK-21 Electric Binding System',
  'profit_change': np.float64(-4573.78),
  'sales_change': np.float64(-2633.99),
  'profit_2020': np.float64(-2929.48),
  'sales_2020': np.float64(1901.99),
  'quantity_2020': 5,
  'orders_2020': 1},
 {'product_name': 'GBC Ibimaster 500 Manual ProClick Binding System',
  'profit_change': np.float64(-4109.29),
  'sales_change': np.float64(-5069.58),
  'profit_2020': np.float64(-1141.47),
  'sales_2020': np.float64(790.98),
  'quantity_2020': 5,
  'orders_2020': 1},
 {'product_name': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind',
  'profit_change': np.float64(-2541.98),
  'sales_change': np.float64(5180.96),
  'profit_2020': np.float64(-1525.19),
  'sales_2020': np.float64(7468.74),
  'quantity_2020': 14,
  'orders_2020': 3},
 {'product_name': 'Ibico Hi-Tech Manual Binding System',
  'profit_change': np.float64(-1113.21),
  'sales_change': np.float64(1243.47),
  'profit_2020': np.float64(-731.98),
 

Turn this into a human-readable insight

In [38]:
def classify_product_risk(product_risks):

    classified = []

    for risk in product_risks:

        profit_loss = abs(risk["profit_change"])
        current_profit = risk["profit_2020"]
        sales_change = risk["sales_change"]

        # Risk score based primarily on profit deterioration
        score = profit_loss

        # Extra risk when the product is currently loss-making
        if current_profit < 0:
            score *= 1.5

        # Extra risk when sales increased but profit declined
        if sales_change > 0 and risk["profit_change"] < 0:
            score *= 1.25

        # Severity thresholds
        if score >= 3000:
            severity = "critical"
        elif score >= 1000:
            severity = "high"
        elif score >= 250:
            severity = "medium"
        else:
            severity = "low"

        classified.append({
            **risk,
            "risk_score": round(score, 2),
            "severity": severity
        })

    return sorted(
        classified,
        key=lambda x: x["risk_score"],
        reverse=True
    )

In [39]:
classified_product_risks = classify_product_risk(
    binder_product_risks
)

classified_product_risks

[{'product_name': 'Ibico EPK-21 Electric Binding System',
  'profit_change': np.float64(-4573.78),
  'sales_change': np.float64(-2633.99),
  'profit_2020': np.float64(-2929.48),
  'sales_2020': np.float64(1901.99),
  'quantity_2020': 5,
  'orders_2020': 1,
  'risk_score': np.float64(6860.67),
  'severity': 'critical'},
 {'product_name': 'GBC Ibimaster 500 Manual ProClick Binding System',
  'profit_change': np.float64(-4109.29),
  'sales_change': np.float64(-5069.58),
  'profit_2020': np.float64(-1141.47),
  'sales_2020': np.float64(790.98),
  'quantity_2020': 5,
  'orders_2020': 1,
  'risk_score': np.float64(6163.94),
  'severity': 'critical'},
 {'product_name': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind',
  'profit_change': np.float64(-2541.98),
  'sales_change': np.float64(5180.96),
  'profit_2020': np.float64(-1525.19),
  'sales_2020': np.float64(7468.74),
  'quantity_2020': 14,
  'orders_2020': 3,
  'risk_score': np.float64(4766.21),
  'severity': 

In [40]:
def generate_product_risk_messages(classified_product_risks):

    messages = []

    for risk in classified_product_risks:

        if (
            risk["sales_change"] > 0
            and risk["profit_change"] < 0
        ):
            message = (
                f"{risk['product_name']} increased sales by "
                f"${risk['sales_change']:,.2f}, but profit declined "
                f"by ${abs(risk['profit_change']):,.2f}."
            )

        elif risk["profit_2020"] < 0:
            message = (
                f"{risk['product_name']} generated a "
                f"${abs(risk['profit_2020']):,.2f} loss in 2020, "
                f"with profit declining by "
                f"${abs(risk['profit_change']):,.2f}."
            )

        else:
            message = (
                f"{risk['product_name']} experienced a profit "
                f"decline of ${abs(risk['profit_change']):,.2f}."
            )

        messages.append({
            "severity": risk["severity"],
            "risk_score": risk["risk_score"],
            "product_name": risk["product_name"],
            "message": message
        })

    return messages

In [41]:
product_risk_messages = generate_product_risk_messages(
    classified_product_risks
)

product_risk_messages

[{'severity': 'critical',
  'risk_score': np.float64(6860.67),
  'product_name': 'Ibico EPK-21 Electric Binding System',
  'message': 'Ibico EPK-21 Electric Binding System generated a $2,929.48 loss in 2020, with profit declining by $4,573.78.'},
 {'severity': 'critical',
  'risk_score': np.float64(6163.94),
  'product_name': 'GBC Ibimaster 500 Manual ProClick Binding System',
  'message': 'GBC Ibimaster 500 Manual ProClick Binding System generated a $1,141.47 loss in 2020, with profit declining by $4,109.29.'},
 {'severity': 'critical',
  'risk_score': np.float64(4766.21),
  'product_name': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind',
  'message': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind increased sales by $5,180.96, but profit declined by $2,541.98.'},
 {'severity': 'high',
  'risk_score': np.float64(2087.27),
  'product_name': 'Ibico Hi-Tech Manual Binding System',
  'message': 'Ibico Hi-Tech Manual Binding System

In [42]:
business_insights = {
    "overall": ...,
    "categories": ...,
    "sub_categories": ...,
    "products": ...
}

In [43]:
classified_product_risks

[{'product_name': 'Ibico EPK-21 Electric Binding System',
  'profit_change': np.float64(-4573.78),
  'sales_change': np.float64(-2633.99),
  'profit_2020': np.float64(-2929.48),
  'sales_2020': np.float64(1901.99),
  'quantity_2020': 5,
  'orders_2020': 1,
  'risk_score': np.float64(6860.67),
  'severity': 'critical'},
 {'product_name': 'GBC Ibimaster 500 Manual ProClick Binding System',
  'profit_change': np.float64(-4109.29),
  'sales_change': np.float64(-5069.58),
  'profit_2020': np.float64(-1141.47),
  'sales_2020': np.float64(790.98),
  'quantity_2020': 5,
  'orders_2020': 1,
  'risk_score': np.float64(6163.94),
  'severity': 'critical'},
 {'product_name': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind',
  'profit_change': np.float64(-2541.98),
  'sales_change': np.float64(5180.96),
  'profit_2020': np.float64(-1525.19),
  'sales_2020': np.float64(7468.74),
  'quantity_2020': 14,
  'orders_2020': 3,
  'risk_score': np.float64(4766.21),
  'severity': 

In [44]:
product_risk_messages

[{'severity': 'critical',
  'risk_score': np.float64(6860.67),
  'product_name': 'Ibico EPK-21 Electric Binding System',
  'message': 'Ibico EPK-21 Electric Binding System generated a $2,929.48 loss in 2020, with profit declining by $4,573.78.'},
 {'severity': 'critical',
  'risk_score': np.float64(6163.94),
  'product_name': 'GBC Ibimaster 500 Manual ProClick Binding System',
  'message': 'GBC Ibimaster 500 Manual ProClick Binding System generated a $1,141.47 loss in 2020, with profit declining by $4,109.29.'},
 {'severity': 'critical',
  'risk_score': np.float64(4766.21),
  'product_name': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind',
  'message': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind increased sales by $5,180.96, but profit declined by $2,541.98.'},
 {'severity': 'high',
  'risk_score': np.float64(2087.27),
  'product_name': 'Ibico Hi-Tech Manual Binding System',
  'message': 'Ibico Hi-Tech Manual Binding System

the final business_insights object 

In [45]:
business_insights = {
    "executive_summary": executive_summary,

    "yearly_insights": yearly_insights,

    "category_insights": classified_category_insights,

    "subcategory_risks": subcategory_risks,

    "product_risks": classified_product_risks,

    "product_messages": product_risk_messages
}

business_insights

{'executive_summary': {'total_sales': np.float64(1565804.32),
  'total_profit': np.float64(175262.11),
  'profit_margin': np.float64(11.19),
  'return_rate': np.float64(4.86),
  'critical_risk_count': 3,
  'warning_risk_count': 1,
  'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
   'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
   'Overall profit margin declined by 5.16 percentage points.'],
  'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']},
 'yearly_insights': [{'insight_type': 'yearly_change',
   'metric': 'Sales',
   'from_year': 2019,
   'to_year': 2020,
   'change_percent': np.float64(77.29)},
  {'insight_type': 'yearly_change',
   'metric': 'Profit',
   'from_year': 2019,
   'to_year': 2020,
   'change_percent': np.float64(14.2)},
  {'insight_type': 'yearly_change',
   'metric': 'Quantity',
   'from_year': 2

In [46]:
business_insights.keys()

dict_keys(['executive_summary', 'yearly_insights', 'category_insights', 'subcategory_risks', 'product_risks', 'product_messages'])

Create a clean executive-level output

In [47]:
def create_executive_insights(business_insights):

    executive = {
        "overall": business_insights["executive_summary"],

        "top_risks": [
            {
                "area": risk["sub_category"],
                "severity": risk["severity"],
                "message": (
                    f"{risk['sub_category']} profit changed by "
                    f"${risk['profit_change_dollars']:,.2f}, "
                    f"while margin changed by "
                    f"{risk['margin_change_pp']:.2f} percentage points."
                )
            }
            for risk in business_insights["subcategory_risks"][:5]
        ],

        "top_product_risks": [
            {
                "product": risk["product_name"],
                "severity": risk["severity"],
                "risk_score": risk["risk_score"],
                "message": next(
                    (
                        item["message"]
                        for item in business_insights["product_messages"]
                        if item["product_name"] == risk["product_name"]
                    ),
                    ""
                )
            }
            for risk in business_insights["product_risks"][:5]
        ]
    }

    return executive

In [48]:
executive_insights = create_executive_insights(
    business_insights
)

executive_insights

{'overall': {'total_sales': np.float64(1565804.32),
  'total_profit': np.float64(175262.11),
  'profit_margin': np.float64(11.19),
  'return_rate': np.float64(4.86),
  'critical_risk_count': 3,
  'warning_risk_count': 1,
  'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
   'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
   'Overall profit margin declined by 5.16 percentage points.'],
  'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']},
 'top_risks': [{'area': 'Binders',
   'severity': 'critical',
   'message': 'Binders profit changed by $-2,545.89, while margin changed by -18.14 percentage points.'},
  {'area': 'Paper',
   'severity': 'warning',
   'message': 'Paper profit changed by $2,969.31, while margin changed by -26.40 percentage points.'},
  {'area': 'Machines',
   'severity': 'critical',
   'message': 'Machine

In [49]:
import numpy as np

def make_json_safe(obj):

    if isinstance(obj, dict):
        return {
            key: make_json_safe(value)
            for key, value in obj.items()
        }

    elif isinstance(obj, list):
        return [
            make_json_safe(item)
            for item in obj
        ]

    elif isinstance(obj, (np.integer,)):
        return int(obj)

    elif isinstance(obj, (np.floating,)):
        return float(obj)

    elif isinstance(obj, (np.bool_,)):
        return bool(obj)

    return obj

In [50]:
business_insights_clean = make_json_safe(
    business_insights
)

executive_insights_clean = make_json_safe(
    executive_insights
)

In [51]:
executive_insights_clean

{'overall': {'total_sales': 1565804.32,
  'total_profit': 175262.11,
  'profit_margin': 11.19,
  'return_rate': 4.86,
  'critical_risk_count': 3,
  'warning_risk_count': 1,
  'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
   'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
   'Overall profit margin declined by 5.16 percentage points.'],
  'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']},
 'top_risks': [{'area': 'Binders',
   'severity': 'critical',
   'message': 'Binders profit changed by $-2,545.89, while margin changed by -18.14 percentage points.'},
  {'area': 'Paper',
   'severity': 'warning',
   'message': 'Paper profit changed by $2,969.31, while margin changed by -26.40 percentage points.'},
  {'area': 'Machines',
   'severity': 'critical',
   'message': 'Machines profit changed by $-5,776.53, while margin cha

In [52]:
def format_profit_change(value):

    if value < 0:
        return f"profit declined by ${abs(value):,.2f}"
    elif value > 0:
        return f"profit increased by ${value:,.2f}"
    else:
        return "profit was unchanged"

In [53]:
def create_executive_insights(business_insights):

    top_risks = []

    for risk in business_insights["subcategory_risks"][:5]:

        profit_text = format_profit_change(
            risk["profit_change_dollars"]
        )

        margin_text = (
            f"margin declined by "
            f"{abs(risk['margin_change_pp']):.2f} percentage points"
            if risk["margin_change_pp"] < 0
            else
            f"margin increased by "
            f"{risk['margin_change_pp']:.2f} percentage points"
        )

        top_risks.append({
            "area": risk["sub_category"],
            "severity": risk["severity"],
            "message": (
                f"{risk['sub_category']}: {profit_text}; "
                f"{margin_text}."
            )
        })

    top_products = []

    for risk in business_insights["product_risks"][:5]:

        matching_message = next(
            (
                item["message"]
                for item in business_insights["product_messages"]
                if item["product_name"] == risk["product_name"]
            ),
            ""
        )

        top_products.append({
            "product": risk["product_name"],
            "severity": risk["severity"],
            "risk_score": risk["risk_score"],
            "message": matching_message
        })

    return {
        "overall": business_insights["executive_summary"],
        "top_risks": top_risks,
        "top_product_risks": top_products
    }

In [54]:
executive_insights = create_executive_insights(
    business_insights
)

executive_insights_clean = make_json_safe(
    executive_insights
)

executive_insights_clean

{'overall': {'total_sales': 1565804.32,
  'total_profit': 175262.11,
  'profit_margin': 11.19,
  'return_rate': 4.86,
  'critical_risk_count': 3,
  'warning_risk_count': 1,
  'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
   'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
   'Overall profit margin declined by 5.16 percentage points.'],
  'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']},
 'top_risks': [{'area': 'Binders',
   'severity': 'critical',
   'message': 'Binders: profit declined by $2,545.89; margin declined by 18.14 percentage points.'},
  {'area': 'Paper',
   'severity': 'warning',
   'message': 'Paper: profit increased by $2,969.31; margin declined by 26.40 percentage points.'},
  {'area': 'Machines',
   'severity': 'critical',
   'message': 'Machines: profit declined by $5,776.53; margin declined by 12.7

In [55]:
import json
from pathlib import Path

output_dir = Path("../reports")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = output_dir / "business_insights.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        executive_insights_clean,
        f,
        indent=4,
        ensure_ascii=False
    )

print(f"Saved to: {output_file}")

Saved to: ..\reports\business_insights.json


In [56]:
from pathlib import Path

output_file.resolve()

WindowsPath('C:/Users/LENOVO/reports/business_insights.json')

In [57]:
output_file.exists()

True

In [58]:
from pathlib import Path

print(Path.cwd())

C:\Users\LENOVO\AI_Data_Insights_Generator


In [59]:
from pathlib import Path
import json

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

output_file = reports_dir / "business_insights.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        executive_insights_clean,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved to:", output_file.resolve())

Saved to: C:\Users\LENOVO\AI_Data_Insights_Generator\reports\business_insights.json


In [60]:
print(output_file.exists())

True


In [61]:
def generate_recommendation(risk):

    area = risk["area"]
    severity = risk["severity"]
    message = risk["message"]

    profit_change = risk.get("profit_change_dollars", 0)
    margin_change = risk.get("margin_change_pp", 0)

    if profit_change < 0 and margin_change < 0:

        recommendation = (
            f"Prioritize {area} for profitability review. "
            "Investigate pricing, discounting, product costs, "
            "and fulfillment expenses."
        )

    elif profit_change > 0 and margin_change < 0:

        recommendation = (
            f"Review margin leakage in {area}. "
            "Although profit increased, the declining margin suggests "
            "that sales growth may be coming at the expense of profitability."
        )

    elif profit_change < 0 and margin_change >= 0:

        recommendation = (
            f"Investigate the decline in {area} profit. "
            "Review sales volume, demand, pricing, and product mix."
        )

    else:

        recommendation = (
            f"Monitor {area} profitability and investigate "
            "any emerging margin pressure."
        )

    return {
        "area": area,
        "severity": severity,
        "risk": message,
        "recommendation": recommendation
    }

In [62]:
recommendations = [
    generate_recommendation({
        "area": risk["sub_category"],
        "severity": risk["severity"],
        "message": (
            f"{risk['sub_category']}: "
            f"profit changed by ${risk['profit_change_dollars']:,.2f}; "
            f"margin changed by {risk['margin_change_pp']:.2f} percentage points."
        ),
        "profit_change_dollars": risk["profit_change_dollars"],
        "margin_change_pp": risk["margin_change_pp"]
    })
    for risk in business_insights["subcategory_risks"][:5]
]

In [63]:
executive_insights_clean["recommendations"] = recommendations

In [64]:
executive_insights_clean

{'overall': {'total_sales': 1565804.32,
  'total_profit': 175262.11,
  'profit_margin': 11.19,
  'return_rate': 4.86,
  'critical_risk_count': 3,
  'warning_risk_count': 1,
  'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
   'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
   'Overall profit margin declined by 5.16 percentage points.'],
  'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']},
 'top_risks': [{'area': 'Binders',
   'severity': 'critical',
   'message': 'Binders: profit declined by $2,545.89; margin declined by 18.14 percentage points.'},
  {'area': 'Paper',
   'severity': 'warning',
   'message': 'Paper: profit increased by $2,969.31; margin declined by 26.40 percentage points.'},
  {'area': 'Machines',
   'severity': 'critical',
   'message': 'Machines: profit declined by $5,776.53; margin declined by 12.7

In [65]:
import json

with open(
    "reports/business_insights.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        executive_insights_clean,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Updated:", output_file.resolve())

Updated: C:\Users\LENOVO\AI_Data_Insights_Generator\reports\business_insights.json


In [66]:
def generate_product_recommendation(risk):

    product = risk["product_name"]
    profit_change = risk["profit_change"]
    profit_2020 = risk["profit_2020"]
    sales_change = risk["sales_change"]
    severity = risk["severity"]

    if profit_2020 < 0 and profit_change < 0:

        recommendation = (
            f"Immediate review recommended for {product}. "
            f"The product generated a loss in 2020 and profit declined "
            f"by ${abs(profit_change):,.2f}. Review pricing, discounts, "
            f"product costs, and consider repricing or discontinuation "
            f"if losses persist."
        )

    elif profit_change < 0 and sales_change > 0:

        recommendation = (
            f"Review {product} for margin leakage. Sales increased while "
            f"profit declined, suggesting that pricing, discounts, or "
            f"costs may be eroding profitability."
        )

    elif profit_change < 0:

        recommendation = (
            f"Investigate the profitability decline of {product}. "
            f"Review demand, pricing, product costs, and sales volume."
        )

    else:

        recommendation = (
            f"Monitor {product} profitability and investigate any "
            f"future deterioration in profit or margin."
        )

    return {
        "product": product,
        "severity": severity,
        "risk_score": risk["risk_score"],
        "recommendation": recommendation
    }

In [67]:
product_recommendations = [
    generate_product_recommendation(risk)
    for risk in business_insights["product_risks"][:10]
]

product_recommendations

[{'product': 'Ibico EPK-21 Electric Binding System',
  'severity': 'critical',
  'risk_score': np.float64(6860.67),
  'recommendation': 'Immediate review recommended for Ibico EPK-21 Electric Binding System. The product generated a loss in 2020 and profit declined by $4,573.78. Review pricing, discounts, product costs, and consider repricing or discontinuation if losses persist.'},
 {'product': 'GBC Ibimaster 500 Manual ProClick Binding System',
  'severity': 'critical',
  'risk_score': np.float64(6163.94),
  'recommendation': 'Immediate review recommended for GBC Ibimaster 500 Manual ProClick Binding System. The product generated a loss in 2020 and profit declined by $4,109.29. Review pricing, discounts, product costs, and consider repricing or discontinuation if losses persist.'},
 {'product': 'Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind',
  'severity': 'critical',
  'risk_score': np.float64(4766.21),
  'recommendation': 'Immediate review recommended f

In [68]:
executive_insights_clean["product_recommendations"] = (
    product_recommendations
)

In [69]:
executive_insights_clean.keys()

dict_keys(['overall', 'top_risks', 'top_product_risks', 'recommendations', 'product_recommendations'])

In [70]:
import json

with open(
    "reports/business_insights.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        executive_insights_clean,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Business insights updated successfully.")

Business insights updated successfully.


In [71]:
[x for x in dir() if not x.startswith("_")]

['In',
 'Out',
 'Path',
 'binder_product_risks',
 'business_insights',
 'business_insights_clean',
 'category_insights',
 'classified_category_insights',
 'classified_product_risks',
 'classified_yearly_insights',
 'classify_category_insights',
 'classify_product_risk',
 'classify_yearly_insights',
 'create_executive_insights',
 'create_insight_messages',
 'df',
 'executive_insights',
 'executive_insights_clean',
 'executive_summary',
 'exit',
 'f',
 'format_profit_change',
 'generate_category_insights',
 'generate_executive_summary',
 'generate_kpi_summary',
 'generate_product_recommendation',
 'generate_product_risk_messages',
 'generate_product_risks',
 'generate_profitability_insight',
 'generate_recommendation',
 'generate_subcategory_insights',
 'generate_subcategory_risks',
 'generate_top_business_risks',
 'generate_yearly_insights',
 'get_ipython',
 'insight_messages',
 'json',
 'kpi_summary',
 'make_json_safe',
 'np',
 'open',
 'os',
 'output_dir',
 'output_file',
 'pd',
 'pro

In [72]:
final_report = {
    "overall": executive_summary,
    "top_risks": top_business_risks,
    "top_product_risks": classified_product_risks,
    "recommendations": recommendations,
    "product_recommendations": product_recommendations
}

final_report

{'overall': {'total_sales': np.float64(1565804.32),
  'total_profit': np.float64(175262.11),
  'profit_margin': np.float64(11.19),
  'return_rate': np.float64(4.86),
  'critical_risk_count': 3,
  'warning_risk_count': 1,
  'critical_risks': ['Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.',
   'Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.',
   'Overall profit margin declined by 5.16 percentage points.'],
  'warning_risks': ['Technology experienced a -2.10 percentage-point margin change and 27.43% profit change.']},
 'top_risks': [{'level': 'category',
   'area': 'Office Supplies',
   'risk_type': 'Category Profitability',
   'severity': 'critical',
   'score': np.float64(11.57),
   'message': 'Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.'},
  {'level': 'category',
   'area': 'Furniture',
   'risk_type': 'Category Profitability',
   'severity': 'crit

In [73]:
import json
from pathlib import Path

report_path = Path("../reports/business_insights.json")

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=4, default=str)

print("Saved to:", report_path.resolve())

Saved to: C:\Users\LENOVO\reports\business_insights.json


In [74]:
print(report_path.exists())
print(report_path)

True
..\reports\business_insights.json


In [75]:
%whos

Variable                          Type             Data/Info
------------------------------------------------------------
Path                              type             <class 'pathlib._local.Path'>
binder_product_risks              list             n=10
business_insights                 dict             n=6
business_insights_clean           dict             n=6
category_insights                 list             n=3
classified_category_insights      list             n=3
classified_product_risks          list             n=10
classified_yearly_insights        list             n=5
classify_category_insights        function         <function classify_catego<...>ts at 0x00000198A4527560>
classify_product_risk             function         <function classify_produc<...>sk at 0x00000198AC588860>
classify_yearly_insights          function         <function classify_yearly<...>ts at 0x00000198A4524AE0>
create_executive_insights         function         <function create_executiv<...>ts at 0x

In [76]:
from pathlib import Path
import json

report_path = Path(r"C:\Users\LENOVO\AI_Data_Insights_Generator\reports\business_insights.json")

print("Report path:", report_path)
print("Exists:", report_path.exists())
print("Size:", report_path.stat().st_size, "bytes")

with open(report_path, "r", encoding="utf-8") as f:
    saved_report = json.load(f)

print("\nJSON loaded successfully!")
print("Top-level sections:", saved_report.keys())

Report path: C:\Users\LENOVO\AI_Data_Insights_Generator\reports\business_insights.json
Exists: True
Size: 9581 bytes

JSON loaded successfully!
Top-level sections: dict_keys(['overall', 'top_risks', 'top_product_risks', 'recommendations', 'product_recommendations'])


In [77]:
print(json.dumps(saved_report, indent=2)[:5000])

{
  "overall": {
    "total_sales": 1565804.32,
    "total_profit": 175262.11,
    "profit_margin": 11.19,
    "return_rate": 4.86,
    "critical_risk_count": 3,
    "warning_risk_count": 1,
    "critical_risks": [
      "Office Supplies experienced a -11.57 percentage-point margin change and 13.33% profit change.",
      "Furniture experienced a -2.40 percentage-point margin change and -56.81% profit change.",
      "Overall profit margin declined by 5.16 percentage points."
    ],
    "warning_risks": [
      "Technology experienced a -2.10 percentage-point margin change and 27.43% profit change."
    ]
  },
  "top_risks": [
    {
      "area": "Binders",
      "severity": "critical",
      "message": "Binders: profit declined by $2,545.89; margin declined by 18.14 percentage points."
    },
    {
      "area": "Paper",
      "severity": "warning",
      "message": "Paper: profit increased by $2,969.31; margin declined by 26.40 percentage points."
    },
    {
      "area": "Machines